In [1]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings
warnings.simplefilter("ignore", FutureWarning)

import os
import pandas as pd
import numpy as np
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

#Full data
#train_df_full = "/mnt/Internal/MedImage/chexpert_balanced_for_training_2000_per_label_dis+demog+age.csv" #used for real/real training for both Densenet/Resnet
train_df_full = "/mnt/Internal/MedImage/chexpert_1000_per_disease.csv" # Used for Generated + Real for both Densenet/Resnet
train_df_full = pd.read_csv(train_df_full)
#train_df_full = pd.get_dummies(train_df_full, columns=["GENDER", "PRIMARY_RACE"])
#train_df_full.columns.tolist()

# Convert categorical columns to integers in train_df_full

# # Race
# train_df_full['PRIMARY_RACE_Asian'] = train_df_full['PRIMARY_RACE_Asian'].astype(int)
# train_df_full['PRIMARY_RACE_Asian - Historical Conv'] = train_df_full['PRIMARY_RACE_Asian - Historical Conv'].astype(int)
# train_df_full['PRIMARY_RACE_Asian, Hispanic'] = train_df_full['PRIMARY_RACE_Asian, Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_Asian, non-Hispanic'] = train_df_full['PRIMARY_RACE_Asian, non-Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_Black or African American'] = train_df_full['PRIMARY_RACE_Black or African American'].astype(int)
# train_df_full['PRIMARY_RACE_Black, Hispanic'] = train_df_full['PRIMARY_RACE_Black, Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_Black, non-Hispanic'] = train_df_full['PRIMARY_RACE_Black, non-Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_White'] = train_df_full['PRIMARY_RACE_White'].astype(int)
# train_df_full['PRIMARY_RACE_White or Caucasian'] = train_df_full['PRIMARY_RACE_White or Caucasian'].astype(int)
# train_df_full['PRIMARY_RACE_White, Hispanic'] = train_df_full['PRIMARY_RACE_White, Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_White, non-Hispanic'] = train_df_full['PRIMARY_RACE_White, non-Hispanic'].astype(int)


# # Gender
# train_df_full['GENDER_Male'] = train_df_full['GENDER_Male'].astype(int)
# train_df_full['GENDER_Female'] = train_df_full['GENDER_Female'].astype(int)

# Display the updated DataFrame
train_df_full.head()

train_df_full.replace(-1, 1, inplace=True)

train_df_full.replace(np.nan, 0, inplace=True)

# # Training Data
# train_df = train_df_full[1:170000]
# #Validation Data
# valid_df = train_df_full[170001:190000]

# Training Data
train_df = train_df_full[0:30265]

train_image_root = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"

In [2]:
ls "/mnt/Internal/MedImage/chexpert_1000_per_disease.csv"

/mnt/Internal/MedImage/chexpert_1000_per_disease.csv


### For validation

In [3]:
import warnings
warnings.filterwarnings("ignore", category=DeprecationWarning)

import warnings
warnings.simplefilter("ignore", FutureWarning)

import os
import pandas as pd
import numpy as np
from torchvision import transforms
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
import torch
import torch.nn as nn
import torchvision.models as models
from sklearn.preprocessing import OneHotEncoder
from sklearn.metrics import roc_auc_score

#Full data
train_df_full1 = pd.read_csv("/mnt/Internal/MedImage/chexpert_balanced_for_training_252_per_label_dis+demog+age.csv") # This set will be used for all models validation

In [4]:

# Convert categorical columns to integers in train_df_full
# # Race
# train_df_full['PRIMARY_RACE_Asian'] = train_df_full['PRIMARY_RACE_Asian'].astype(int)
# train_df_full['PRIMARY_RACE_Asian - Historical Conv'] = train_df_full['PRIMARY_RACE_Asian - Historical Conv'].astype(int)
# train_df_full['PRIMARY_RACE_Asian, Hispanic'] = train_df_full['PRIMARY_RACE_Asian, Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_Asian, non-Hispanic'] = train_df_full['PRIMARY_RACE_Asian, non-Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_Black or African American'] = train_df_full['PRIMARY_RACE_Black or African American'].astype(int)
# train_df_full['PRIMARY_RACE_Black, Hispanic'] = train_df_full['PRIMARY_RACE_Black, Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_Black, non-Hispanic'] = train_df_full['PRIMARY_RACE_Black, non-Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_White'] = train_df_full['PRIMARY_RACE_White'].astype(int)
# train_df_full['PRIMARY_RACE_White or Caucasian'] = train_df_full['PRIMARY_RACE_White or Caucasian'].astype(int)
# train_df_full['PRIMARY_RACE_White, Hispanic'] = train_df_full['PRIMARY_RACE_White, Hispanic'].astype(int)
# train_df_full['PRIMARY_RACE_White, non-Hispanic'] = train_df_full['PRIMARY_RACE_White, non-Hispanic'].astype(int)


# # Gender
# train_df_full['GENDER_Male'] = train_df_full['GENDER_Male'].astype(int)
# train_df_full['GENDER_Female'] = train_df_full['GENDER_Female'].astype(int)['ETHNICITY_Patient Refused'] = train_df_full1['ETHNICITY_Patient Refused'].astype(int)

# # Gender
# train_df_full1['GENDER_Male'] = train_df_full1['GENDER_Male'].astype(int)
# train_df_full1['GENDER_Female'] = train_df_full1['GENDER_Female'].astype(int)

# Display the updated DataFrame
train_df_full1.head()

train_df_full1.replace(-1, 1, inplace=True)

train_df_full1.replace(np.nan, 0, inplace=True)

valid_df = train_df_full1[0:7811]

valid_image_root = "/mnt/Internal/MedImage/CheXpert Dataset/unzip_chexpert_images/CheXpert-v1.0/train"

In [5]:
import os
import torch
import numpy as np
import pandas as pd
from PIL import Image, UnidentifiedImageError
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

# Step 1: Create age group labels
def categorize_age(age):
    if age <= 30:
        return 'AGE_GROUP_AGE_0_30'
    elif age <= 50:
        return 'AGE_GROUP_AGE_31_50'
    elif age <= 70:
        return 'AGE_GROUP_AGE_51_70'
    else:
        return 'AGE_GROUP_AGE_71_plus'

# Apply age group and one-hot encode
for df in [train_df, valid_df]:
    df['AGE_GROUP'] = df['Age'].apply(categorize_age)

train_df = pd.get_dummies(train_df, columns=['AGE_GROUP'])
valid_df = pd.get_dummies(valid_df, columns=['AGE_GROUP'])

# Ensure both dataframes have all age group columns
age_group_cols = ['AGE_GROUP_AGE_0_30', 'AGE_GROUP_AGE_31_50', 'AGE_GROUP_AGE_51_70', 'AGE_GROUP_AGE_71_plus']
for col in age_group_cols:
    for df in [train_df, valid_df]:
        if col not in df.columns:
            df[col] = 0

# Step 2: Define transformations for X-ray images
transform = transforms.Compose([
    transforms.Resize((320, 320)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])  # Normalization for ImageNet
])

# Step 3: Define Dataset class
class CheXpertDataset(Dataset):
    def __init__(self, dataframe, transform=None, image_root=None):
        self.dataframe = dataframe
        self.transform = transform
        self.image_root = image_root

        self.label_cols = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion', 
          'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax', 
          'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
          'GENDER_Female', 'GENDER_Male',
          'PRIMARY_RACE_Asian', 'PRIMARY_RACE_Asian - Historical Conv', 'PRIMARY_RACE_Asian, Hispanic',
          'PRIMARY_RACE_Asian, non-Hispanic', 'PRIMARY_RACE_Black or African American', 'PRIMARY_RACE_Black, Hispanic', 'PRIMARY_RACE_Black, non-Hispanic',
          'PRIMARY_RACE_White', 'PRIMARY_RACE_White or Caucasian', 'PRIMARY_RACE_White, Hispanic', 'PRIMARY_RACE_White, non-Hispanic',
          'AGE_GROUP_AGE_0_30', 'AGE_GROUP_AGE_31_50', 'AGE_GROUP_AGE_51_70', 'AGE_GROUP_AGE_71_plus']

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, idx):
        item = self.dataframe.iloc[idx]
        img_path = item['Path']
        img_path = img_path.replace("CheXpert-v1.0/train/", "")
        img_path = os.path.join(self.image_root, img_path)

        try:
            image = Image.open(img_path).convert("RGB")
        except (FileNotFoundError, IOError, UnidentifiedImageError):
            return None

        if self.transform:
            image = self.transform(image)

        image = np.array(image)
        label = np.array(item[self.label_cols])

        return image, label

# Step 4: Custom collate function to handle None values
def collate_fn(batch):
    batch = [b for b in batch if b is not None]
    if len(batch) == 0:
        return torch.empty(0), torch.empty(0)

    images, labels = zip(*batch)
    images = np.array(images, dtype=np.float32)
    labels = np.array(labels, dtype=np.float32)
    labels = np.reshape(labels, (labels.shape[0], len(labels[0])))

    images = torch.tensor(images)
    labels = torch.tensor(labels)

    return images, labels

# Step 6: Create DataLoaders
train_dataset = CheXpertDataset(train_df, transform=transform, image_root=train_image_root)
train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True, collate_fn=collate_fn)

valid_dataset = CheXpertDataset(valid_df, transform=transform, image_root=valid_image_root)
valid_loader = DataLoader(valid_dataset, batch_size=16, shuffle=False, collate_fn=collate_fn)

# Step 7: Debugging
print(f"Training Samples: {len(train_dataset)}")
print(f"Validation Samples: {len(valid_dataset)}")


Training Samples: 14000
Validation Samples: 6766


/tmp/ipykernel_2857060/4154743603.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AGE_GROUP'] = df['Age'].apply(categorize_age)
/tmp/ipykernel_2857060/4154743603.py:22: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  df['AGE_GROUP'] = df['Age'].apply(categorize_age)


### DensNet121

In [6]:
# # Step 6: Load a pretrained DenseNet-121 model
# model = models.densenet121(pretrained=True)

# # Step 7: Modify the final layer for binary classification
# num_features = model.classifier.in_features
# model.classifier = nn.Linear(num_features, 31)

# # Step 8: Send model to GPU if availablme
# device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
# model = model.to(device)

# # Step 9: Set up loss function and optimizer
# criterion = nn.BCEWithLogitsLoss()  # For multi-label classification
# optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


### RESNET50

In [7]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torchvision import models

# Step 6: Load a pretrained ResNet-50 model
model = models.resnet50(pretrained=True)

# Step 7: Modify the final layer for 31-class multi-label classification
num_features = model.fc.in_features
model.fc = nn.Linear(num_features,14)  # 31 labels including diseases, gender, race, age groups

# Step 8: Send model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model = model.to(device)

# Step 9: Set up loss function and optimizer
criterion = nn.BCEWithLogitsLoss()  # Suitable for multi-label classification
optimizer = torch.optim.Adam(model.parameters(), lr=1e-4)


/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/torchvision/models/_utils.py:208: UserWarning: The parameter 'pretrained' is deprecated since 0.13 and may be removed in the future, please use 'weights' instead.
  warnings.warn(
/home/dawood/lab2_rotaion/.conda/lib/python3.12/site-packages/torchvision/models/_utils.py:223: UserWarning: Arguments other than a weight enum or `None` for 'weights' are deprecated since 0.13 and may be removed in the future. The current behavior is equivalent to passing `weights=ResNet50_Weights.IMAGENET1K_V1`. You can also use `weights=ResNet50_Weights.DEFAULT` to get the most up-to-date weights.
  warnings.warn(msg)


### Training and Validation for All Selected Metrics

In [9]:
ls "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/resnet50_Real_vs_real_training_for_14_disease"

In [10]:
import os
import numpy as np
import torch
import torch.nn.functional as F
from sklearn.metrics import roc_auc_score, precision_score, recall_score, confusion_matrix, brier_score_loss

# Define CheXpert labels
chexpert_labels = [
    'No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity', 'Lung Lesion',
    'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
    'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices',
]

checkpoint_dir = "/mnt/Internal/MedImage/CheXpert Dataset/Lab_Rotation_2/resnet50_Real_vs_real_training_for_14_disease"
os.makedirs(checkpoint_dir, exist_ok=True)

num_epochs = 5
checkpoint_interval = 4800
total_iterations = 0
resume_epoch = None

if resume_epoch is not None:
    checkpoint_path = os.path.join(checkpoint_dir, f'model_epoch_{resume_epoch}.pth')
    if os.path.exists(checkpoint_path):
        model.load_state_dict(torch.load(checkpoint_path))
        print(f'Resuming training from {checkpoint_path}')
    else:
        print(f'Checkpoint {checkpoint_path} not found. Starting from scratch.')

auc_per_epoch = []
bce_loss_per_epoch = []

for epoch in range(resume_epoch if resume_epoch else 0, num_epochs):
    model.train()
    total_loss = 0

    for images, labels in train_loader:
        if images.size(0) == 0:
            continue

        images, labels = images.to(device), labels.to(device).float()
        outputs = model(images)
        loss = criterion(outputs, labels)

        total_loss += loss.item()
        total_iterations += 1

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        if total_iterations % checkpoint_interval == 0:
            checkpoint_path = os.path.join(checkpoint_dir, f'checkpoint_epoch_{epoch + 1}_iter_{total_iterations}.pth')
            torch.save(model.state_dict(), checkpoint_path)
            print(f'Checkpoint saved at: {checkpoint_path}')

    avg_loss = total_loss / len(train_loader) if len(train_loader) > 0 else 0
    print(f'Epoch [{epoch + 1}/{num_epochs}], Average Loss: {avg_loss:.4f}')

    model.eval()
    all_outputs = []
    all_labels = []
    all_losses = []

    with torch.no_grad():
        for images, labels in valid_loader:
            if images.size(0) == 0:
                continue

            images, labels = images.to(device), labels.to(device).float()
            outputs = model(images)
            loss = F.binary_cross_entropy_with_logits(outputs, labels)
            all_losses.append(loss.item())

            all_outputs.append(outputs.cpu().numpy())
            all_labels.append(labels.cpu().numpy())

    all_outputs = np.concatenate(all_outputs, axis=0)
    all_labels = np.concatenate(all_labels, axis=0)
    all_outputs_sigmoid = 1 / (1 + np.exp(-all_outputs))

    auc_scores = []
    precision_scores = []
    tpr_scores = []
    fpr_scores = []
    bce_scores = []
    ece_scores = []
    error_rates = []

    num_classes = all_labels.shape[1]

    for i in range(num_classes):
        y_true = all_labels[:, i]
        y_pred = all_outputs_sigmoid[:, i]
        y_pred_label = (y_pred >= 0.5).astype(int)

        try:
            auc = roc_auc_score(y_true, y_pred)
            precision = precision_score(y_true, y_pred_label, zero_division=0)
            recall = recall_score(y_true, y_pred_label, zero_division=0)
            tn, fp, fn, tp = confusion_matrix(y_true, y_pred_label, labels=[0, 1]).ravel()
            fpr = fp / (fp + tn) if (fp + tn) > 0 else 0.0
            ece = brier_score_loss(y_true, y_pred)
            bce = F.binary_cross_entropy(torch.tensor(y_pred), torch.tensor(y_true)).item()
            error_rate = (fp + fn) / (tp + tn + fp + fn)
        except ValueError:
            auc = float('nan')
            precision = float('nan')
            recall = float('nan')
            fpr = float('nan')
            ece = float('nan')
            bce = float('nan')
            error_rate = float('nan')

        auc_scores.append(auc)
        precision_scores.append(precision)
        tpr_scores.append(recall)
        fpr_scores.append(fpr)
        ece_scores.append(ece)
        bce_scores.append(bce)
        error_rates.append(error_rate)

        label_name = chexpert_labels[i] if i < len(chexpert_labels) else f'Label_{i}'
        print(f'{label_name}: AUC={auc:.4f}, Precision={precision:.4f}, TPR={recall:.4f}, '
              f'FPR={fpr:.4f}, ECE={ece:.4f}, BCE={bce:.4f}, ErrorRate={error_rate:.4f}')

    auc_per_epoch.append(auc_scores)
    bce_loss_per_epoch.append(np.mean(bce_scores))

    avg_auc = np.nanmean(auc_scores)
    print(f'Average AUC for epoch {epoch + 1}: {avg_auc:.4f}')
    print(f'Average BCE for epoch {epoch + 1}: {np.mean(bce_scores):.4f}')

    epoch_checkpoint_path = os.path.join(checkpoint_dir, f'model_epoch_{epoch + 1}.pth')
    torch.save(model.state_dict(), epoch_checkpoint_path)
    print(f'Model saved at the end of epoch {epoch + 1}: {epoch_checkpoint_path}')

print("Training complete!")

for epoch_idx in range(num_epochs):
    print(f'\nEpoch {epoch_idx + 1} Validation Summary:')
    print(f'AUC scores: {auc_per_epoch[epoch_idx]}')
    print(f'Average BCE Loss: {bce_loss_per_epoch[epoch_idx]:.4f}')

Epoch [1/5], Average Loss: 0.4439
No Finding: AUC=0.8458, Precision=0.5234, TPR=0.1124, FPR=0.0099, ECE=0.0660, BCE=0.2290, ErrorRate=0.0872
Enlarged Cardiomediastinum: AUC=0.6082, Precision=0.0000, TPR=0.0000, FPR=0.0000, ECE=0.1113, BCE=0.3785, ErrorRate=0.1299
Cardiomegaly: AUC=0.7923, Precision=0.6580, TPR=0.2000, FPR=0.0212, ECE=0.1132, BCE=0.3686, ErrorRate=0.1530
Lung Opacity: AUC=0.7095, Precision=0.6274, TPR=0.8801, FPR=0.6042, ECE=0.2243, BCE=0.6430, ErrorRate=0.3445
Lung Lesion: AUC=0.7417, Precision=0.0000, TPR=0.0000, FPR=0.0000, ECE=0.0629, BCE=0.2347, ErrorRate=0.0715
Edema: AUC=0.7859, Precision=0.7250, TPR=0.2604, FPR=0.0485, ECE=0.1871, BCE=0.5611, ErrorRate=0.2759
Consolidation: AUC=0.6568, Precision=0.3261, TPR=0.2474, FPR=0.1406, ECE=0.1794, BCE=0.5354, ErrorRate=0.2725
Pneumonia: AUC=0.7188, Precision=0.4375, TPR=0.0077, FPR=0.0015, ECE=0.1068, BCE=0.3576, ErrorRate=0.1343
Atelectasis: AUC=0.6695, Precision=0.4903, TPR=0.4849, FPR=0.2688, ECE=0.2138, BCE=0.6144, E